# Voice cloning for Sheng (Swahili + English) on a free Colab GPU

Fine-tune **Chatterbox Multilingual** (0.5 B `T3` LM, `sw` = Swahili is a supported
language) so it clones one Sheng speaker. The base model already clones any voice
from a ~6 s reference clip; the fine-tune is what makes the *accent and delivery*
of that speaker stick.

Pipeline: **measure -> tune -> measure again**. Every stage resumes after a
disconnect, and nothing is wasted if the VM dies.

Runtime: **L4 (24 GB)**. `Runtime > Change runtime type > L4 GPU`.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Drive (artifacts live there, the VM does not)

Checkpoints are ~1 GB each; a free Colab VM is deleted without warning.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['SHENG_VOICECLONE_ARTIFACTS'] = '/content/drive/MyDrive/voice-clone-sheng'

## 3. Code + dependencies

`chatterbox-tts` does the synthesis, `transformers` gives the ASR judge,
`speechbrain` gives the speaker-similarity judge. The trainer comes from the
sibling `finetune-chatterbox` project.

In [ ]:
!git clone https://github.com/<you>/audio-speech-ai.git /content/audio-speech-ai
%cd /content/audio-speech-ai/voice-clone
!pip install -q -e .
!pip install -q -U chatterbox-tts transformers speechbrain
!pip install -q -e ../finetune-chatterbox     # the T3 trainer this driver calls

## 4. Environment + GPU profile

In [ ]:
!sheng-voiceclone setup

## 5. Data: dataset -> 16 kHz WAVs, splits, one reference clip

`42 clips x 30 s` of one Sheng speaker. The reference clip is taken from the
**validation** split, never from training, so a similarity score later is honest.

In [ ]:
!sheng-voiceclone data

## 6. Base model snapshot (skippable if you already have it on Drive)

Also fetch `ve.pt` / `s3gen.pt` / `conds.pt`; the fine-tune only touches `t3`.

In [ ]:
!sheng-voiceclone model

## 7. Baseline: clone with the **untuned** model and score it

This is the number to beat. It answers "do I even need to train?" - sometimes the
answer is no, and you save the GPU hours.

~7 prompts (Sheng / clean Swahili / English) x (CER via Whisper + speaker cosine
via ECAPA).

In [ ]:
!sheng-voiceclone baseline --prompts sheng,swahili

### Look at the baseline before spending any GPU time

Open `scores/baseline.md`. If it already says `ok` on the Sheng prompts and the
speaker similarity is high, fine-tuning will only trade one failure mode for
another - read `compare` at the end and be willing to ship the base model.

In [ ]:
from pathlib import Path
import os
root = Path(os.environ['SHENG_VOICECLONE_ARTIFACTS'])
print((root / 'scores' / 'baseline.md').read_text())

## 8. Fine-tune

LoRA is the default recommendation for 21 minutes of audio (the toolkit guidance
is: LoRA for anything under ~10 h). Pass `--lora` to use it, or leave it off for a
full T3 fine-tune at a low LR.

`--max-hours 8` stops gracefully before Colab kills the VM; the trainer writes a
checkpoint on SIGINT.

In [ ]:
!sheng-voiceclone train --profile l4 --steps 600 --lr 5e-5 --lora --max-hours 8

## 9. Score the checkpoint with the *same* prompts

Same text, same reference clip, same judges - otherwise the comparison is
meaningless.

In [ ]:
!sheng-voiceclone eval --prompts sheng,swahili

## 10. Verdict: did fine-tuning actually help?

`KEEP` / `MIXED` / `REJECT` / `NO GAIN`, from CER, WER and speaker similarity
against the baseline.

In [ ]:
!sheng-voiceclone compare

## 11. Listen (the only judge that matters)

In [ ]:
from IPython.display import Audio, display
import os
from pathlib import Path

root = Path(os.environ['SHENG_VOICECLONE_ARTIFACTS'])
for label, d in (('baseline', 'baseline'), ('tuned', sorted((root/'evals').glob('tuned_*'))[-1].name if list((root/'evals').glob('tuned_*')) else None)):
    if not d:
        continue
    for wav in sorted((root / 'evals' / d).glob('*.wav'))[:3]:
        print(f'{label}: {wav.name}')
        display(Audio(str(wav)))

## 12. After a disconnect

Same order, same commands. Finished stages are skipped and training resumes from
`latest`:

```python
%cd /content/audio-speech-ai/voice-clone
!git pull
!pip install -q -e . && pip install -q -e ../finetune-chatterbox
!sheng-voiceclone status
!sheng-voiceclone all --resume --max-hours 8
```

In [ ]:
!sheng-voiceclone status

## Honest expectations

* **21 minutes** is a *voice* fine-tune: timbre, accent, delivery. It will not
  teach new Sheng slang the base model has never seen - keep prompts in the
  dataset's register.
* CER on code-switched Sheng is high even for the base model (Swahili is
  agglutinative; code-switches confuse Whisper). Only the **delta** matters.
* If the output turns to noise, halve the LR or stop earlier - overfitting is the
  main failure mode at this data size.